---

## 1. Allocation Algorithm (Steps)

The allocation algorithm turns a target EV into concrete shutter speed, aperture, and ISO under Constraints and a user **preference** (shutter | aperture | iso | balanced). Implemented in src/allocation/allocateSettings.ts.

### Step 1: Quantize target EV

The allocator works in discrete EV steps (e.g. 1/3 EV). The incoming target EV is rounded to the nearest step so all downstream math uses a single quantized value.

- **Formula:** quantizedEV = round(targetEV / quantizationStep) * quantizationStep
- If |quantizedEV - targetEV| > 0.001, the log records quantizationApplied: true so the UI can explain that the target was adjusted to a valid step.
- **Parameters:** targetEV (from AE: base EV + ΔEV, then clamped); constraints.quantizationStep (e.g. 1/3).

### Step 2: Choose preference and target EV split

The user's **preference** fixes how the total EV is allocated across shutter, aperture, and ISO. Each preference corresponds to a target split of EV contributions ($e_T$, $e_N$, $e_S$) or a priority order.

| Preference | Strategy |
|------------|----------|
| **shutter** | Give shutter the full EV first ($e_T$ = quantizedEV). Remainder split 60% aperture / 40% ISO. |
| **aperture** | Give aperture the full EV first ($e_N$ = quantizedEV). Remainder split 60% shutter / 40% ISO. |
| **iso** | Fix ISO at base ($e_S$ = 0). Split full EV 50% shutter / 50% aperture (minimize ISO). |
| **balanced** | Split evenly: $e_T = e_N = e_S = \text{quantizedEV} / 3$. |

**Base reference values:** BASE_SHUTTER = 1/60, BASE_APERTURE = 2.8, BASE_ISO = 100

### Step 3: Compute settings and apply constraints

**EV ↔ settings formulas:**

- **Shutter:** $T = \text{BASE\_SHUTTER} \cdot 2^{e_T}$; inverse $e_T = \log_2(T / \text{BASE\_SHUTTER})$
- **Aperture:** $N = \text{BASE\_APERTURE} \cdot (\sqrt{2})^{-e_N}$; inverse uses √2 stops
- **ISO:** $S = \text{BASE\_ISO} \cdot 2^{e_S}$

**Clamping:** Shutter to [shutterMin, shutterMax], aperture to [apertureMin, apertureMax], ISO to [ISO_MIN=100, isoMax].

**Quantization:** Aperture → nearest standard f-stop in {1, 1.4, 2, 2.8, 4, 5.6, 8, 11, 16, 22, 32}. ISO → nearest standard stop in {100, 200, 400, …}. Shutter is only clamped (no discrete stop list).

### Step 4: Record constraint hits and finalize

- **Constraint hits:** For each of shutter, aperture, and ISO, if the value equals the min/max bound, push shutter_min, shutter_max, aperture_min, aperture_max, or iso_max onto log.constraintHits.
- **Sanitize:** clampFinite(x, min, max) ensures the value is finite and positive.
- **EV breakdown:** Compute shutterEV, apertureEV, isoEV from the inverse formulas; totalEV = shutterEV + apertureEV + isoEV. Store in log.evBreakdown.

**Output:** settings: { shutterSeconds, aperture, iso } and log: { constraintHits, quantizationApplied, preference, evBreakdown? }

### Auxiliary: evRangeFromConstraints

Sweeps EV from -10 to +15 in quantizationStep steps, calls allocateSettings(ev, constraints, preference) for each EV, and returns the min and max EV where log.constraintHits.length === 0. This range is used to clamp the target EV before calling allocateSettings.

---

## 2. Flowcharts

Three flowcharts: AE (EV selection) → Allocation → simulateForward. Mermaid diagrams below can be rendered in viewers that support Mermaid (e.g. GitHub, VS Code with extension).

### 2.1 AE (EV selection)

```mermaid
flowchart TB
    classDef input fill:#e3f2fd,stroke:#1976d2
    classDef process fill:#e8f5e9,stroke:#388e3c
    classDef decision fill:#fff3e0,stroke:#f57c00
    classDef output fill:#f3e5f5,stroke:#7b1fa2
    classDef relax fill:#ffebee,stroke:#c62828

    IMG[Scene image]
    W[Metering weights]
    P[AEPriorities]
    EV_RANGE[evRange from evRangeFromConstraints]
    LUM[Compute base luminance]
    HIST_INIT[computeWeightedHistogram base luminance]
    IQR["computeIQRBounds Q1 Q3 lower upper fence"]
    ALGO_TYPE{Algorithm?}
    GLOB["Global: w=1 for in-range pixels"]
    SEM["Semantic/Entropy: w=metering weight for in-range"]
    SAL["Saliency: w = |L-mean| × metering weight for in-range"]
    NORM[Normalize Σw=1]
    SWEEP[EV sweep: scale L×2^EV computeClipping computeWeightedHistogram]
    CANDIDATES[Per EV: highlightClip shadowClip median midtoneError entropy]
    SEL_TYPE{Selection path?}
    E_PICK[Pick argmax entropy no feasibility check]
    FEAS[Feasible = highlightClip ≤ ηh ∧ shadowClip ≤ ηs]
    EMPTY{Feasible non-empty?}
    FEASIBLE_SET[Feasible set non-empty]
    BEST[argmin midtoneError]
    RELAX["Relax: single step over all candidates"]
    R1["highlightRatio = clip/ηh shadowRatio = clip/ηs"]
    R2["penalty = L1/L2/L∞ of ratios"]
    R3["argmin penalty tie-break midtoneError"]
    CHOSEN[chosen candidate EV]
    CLAMP_EV[Clamp to evRange min max]
    OUT[chosenEV]

    IMG --> LUM
    W --> LUM
    P --> EV_RANGE
    LUM --> HIST_INIT
    HIST_INIT --> IQR
    IQR --> ALGO_TYPE
    ALGO_TYPE -->|global| GLOB
    ALGO_TYPE -->|semantic| SEM
    ALGO_TYPE -->|entropy| SEM
    ALGO_TYPE -->|saliency| SAL
    GLOB --> NORM
    SEM --> NORM
    SAL --> NORM
    NORM --> SWEEP
    EV_RANGE --> SWEEP
    SWEEP --> CANDIDATES
    CANDIDATES --> SEL_TYPE
    SEL_TYPE -->|entropy| E_PICK
    SEL_TYPE -->|global/semantic/saliency| FEAS
    E_PICK --> CHOSEN
    FEAS --> EMPTY
    EMPTY -->|Yes| FEASIBLE_SET
    EMPTY -->|No| RELAX
    RELAX --> R1 --> R2 --> R3 --> FEASIBLE_SET
    FEASIBLE_SET --> BEST
    BEST --> CHOSEN
    CHOSEN --> CLAMP_EV
    EV_RANGE --> CLAMP_EV
    CLAMP_EV --> OUT
```

### 2.2 Allocation

```mermaid
flowchart TB
    classDef input fill:#e3f2fd,stroke:#1976d2
    classDef process fill:#e8f5e9,stroke:#388e3c
    classDef decision fill:#fff3e0,stroke:#f57c00
    classDef output fill:#f3e5f5,stroke:#7b1fa2

    CHOSEN_EV[chosenEV from AE]
    BASE[baseEV from current settings]
    EV_RANGE[evRangeFromConstraints]
    TARGET[target EV = baseEV + chosenEV]
    CLAMP_T[Clamp target to evRange]
    CONST[Constraints]
    PREF_IN[Preference]
    QEV[Quantize EV to 1/3 steps]
    PREF{Preference?}
    SHUT[Shutter: full EV to shutter 60/40 remainder]
    APERT[Aperture: full EV to aperture 60/40 remainder]
    ISO_P[ISO: ISO=min 50/50 shutter aperture]
    BAL[Balanced: e_T=e_N=e_S=EV/3]
    EV_TO_T[evToShutter evToAperture evToISO]
    CLAMP[Clamp to shutterMin/Max apertureMin/Max isoMax]
    QUANT[quantizeAperture quantizeISO to standard stops]
    SAFE[clampFinite sanitize]
    HITS[Record constraint hits]
    EV_BD[evBreakdown shutterEV apertureEV isoEV]
    SETTINGS[CameraSettings]
    SIM[simulateForward]

    CHOSEN_EV --> TARGET
    BASE --> TARGET
    CONST --> EV_RANGE
    PREF_IN --> EV_RANGE
    TARGET --> CLAMP_T
    EV_RANGE --> CLAMP_T
    CLAMP_T --> QEV
    CONST --> QEV
    PREF_IN --> QEV
    QEV --> PREF
    PREF --> SHUT
    PREF --> APERT
    PREF --> ISO_P
    PREF --> BAL
    SHUT --> EV_TO_T
    APERT --> EV_TO_T
    ISO_P --> EV_TO_T
    BAL --> EV_TO_T
    EV_TO_T --> CLAMP
    CLAMP --> QUANT
    QUANT --> SAFE
    SAFE --> HITS
    HITS --> EV_BD
    EV_BD --> SETTINGS
    SETTINGS --> SIM
```

### 2.3 simulateForward

```mermaid
flowchart TB
    classDef input fill:#e3f2fd,stroke:#1976d2
    classDef process fill:#e8f5e9,stroke:#388e3c
    classDef decision fill:#fff3e0,stroke:#f57c00
    classDef output fill:#f3e5f5,stroke:#7b1fa2

    SCENE[Scene: image illumination subjectMask]
    SETTINGS_IN[CameraSettings: shutter aperture iso]
    SIM_PARAMS[SimParams: fullWell readNoise dofStrength motionEnabled motionThreshold motionDirection]
    ILLUM[Step 1: Apply illumination scalar to RGB]
    EV_CALC[evCurrent evRef from settings and EXIF]
    EXP_SCALE[exposureScale = 2^evCurrent - evRef]
    SCALE_RGB[Scale RGB by exposureScale]
    VAR[Step 3: variance = fullWell/ISO + readNoise²]
    ISO_GAIN[isoGain noiseBoost from base ISO]
    NOISE[Add shot noise + read noise to each pixel]
    DOF_CHECK{dofStrength > 0?}
    KERNEL[createGaussianKernel blurSigma from aperture]
    CONV[applyConvolution subjectMask keeps subject sharp]
    MOTION_CHECK{motionEnabled and shutter > threshold?}
    MOTION_DIR{motionDirection?}
    MOTION_BLUR[applyMotionBlur directional]
    MOTION_ISO[applyMotionBlur isotropic multi-direction]
    CLIP_MASKS[Compute highlightClipMask shadowClipMask luminance ≥1 or ≤ε]
    OUT_SIM[SimOutput: image highlightClipMask shadowClipMask]

    SCENE --> ILLUM
    SETTINGS_IN --> EV_CALC
    SETTINGS_IN --> VAR
    SETTINGS_IN --> NOISE
    SIM_PARAMS --> VAR
    SIM_PARAMS --> DOF_CHECK
    SIM_PARAMS --> MOTION_CHECK
    ILLUM --> EV_CALC
    EV_CALC --> EXP_SCALE
    EXP_SCALE --> SCALE_RGB
    SCALE_RGB --> VAR
    VAR --> ISO_GAIN
    ISO_GAIN --> NOISE
    NOISE --> DOF_CHECK
    DOF_CHECK -->|Yes| KERNEL
    DOF_CHECK -->|No| MOTION_CHECK
    KERNEL --> CONV
    CONV --> MOTION_CHECK
    MOTION_CHECK -->|Yes| MOTION_DIR
    MOTION_CHECK -->|No| CLIP_MASKS
    MOTION_DIR -->|directional| MOTION_BLUR
    MOTION_DIR -->|isotropic| MOTION_ISO
    MOTION_BLUR --> CLIP_MASKS
    MOTION_ISO --> CLIP_MASKS
    CLIP_MASKS --> OUT_SIM
```